# Proyecto final: Todo modelo es un voto ponderado
**Análisis y Ciencia de Datos — UNIS**  ·  Grupo Bikeshare: Rodolfo Flores, Nohel Estrada, Humberto Najar

Casi todos los modelos del curso predicen como
$$\hat y(x) = \sum_{i=1}^n w_i(x)\, y_i .$$
En este notebook le sacamos los pesos a OLS, ridge, $k$-NN, un árbol y un *random forest*,
comprobamos que reproducen `.predict()`, medimos cuánta evidencia respalda cada predicción
y cuánto cuesta la flexibilidad. La Parte 1 usa California Housing y las Partes 2–4 usan Bikeshare.

Las funciones de pesos están en `pesos.py` (mismo directorio).

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error, silhouette_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import pesos as P

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
FIG = "../figuras"
RS = 0

## Preparación

### 0.1 Partición y estandarización
80/20 con `random_state=0`; el `StandardScaler` se ajusta **solo** con entrenamiento.

In [2]:
# California Housing (Parte 1)
cal = fetch_california_housing(as_frame=True)
Xc_raw_tr, Xc_raw_te, yc_tr, yc_te = train_test_split(cal.data, cal.target, test_size=0.2, random_state=RS)
esc_c = StandardScaler().fit(Xc_raw_tr)
Xc_tr, Xc_te = esc_c.transform(Xc_raw_tr), esc_c.transform(Xc_raw_te)
yc_tr, yc_te = yc_tr.to_numpy(), yc_te.to_numpy()
print("California:", Xc_tr.shape, Xc_te.shape)

California: (16512, 8) (4128, 8)


In [3]:
# Bikeshare (Partes 2 a 5)
bike = pd.read_csv("../data/Bikeshare.csv")
meses = {m: i + 1 for i, m in enumerate(["Jan", "Feb", "March", "April", "May", "June", "July",
                                         "Aug", "Sept", "Oct", "Nov", "Dec"])}
clima = {"clear": 1, "cloudy/misty": 2, "light rain/snow": 3, "heavy rain/snow": 4}
print("meses en el archivo:", bike.mnth.unique())
bike["mnth"] = bike["mnth"].map(meses)
bike["weathersit"] = bike["weathersit"].map(clima)
bike["hr"] = bike["hr"].astype(int)
assert bike[["mnth", "weathersit"]].notna().all().all()

VARS = ["day", "hr", "weekday", "workingday", "holiday", "temp", "hum", "windspeed", "mnth", "weathersit"]
Xb = bike[VARS].astype(float)
yb = np.log1p(bike["bikers"]).to_numpy()
Xb_raw_tr, Xb_raw_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=RS)
esc_b = StandardScaler().fit(Xb_raw_tr)
Xb_tr, Xb_te = esc_b.transform(Xb_raw_tr), esc_b.transform(Xb_raw_te)
bike_tr, bike_te = bike.loc[Xb_raw_tr.index], bike.loc[Xb_raw_te.index]
n_b = len(Xb_tr)
print("Bikeshare:", Xb_tr.shape, Xb_te.shape)

meses en el archivo: <StringArray>
['Jan', 'Feb', 'March', 'April', 'May', 'June', 'July', 'Aug', 'Sept', 'Oct', 'Nov', 'Dec']
Length: 12, dtype: str
Bikeshare: (6916, 10) (1729, 10)


### 0.2 Taiwan
No aplica a nuestro grupo (usamos Bikeshare), pero la idea es útil: si $w_i(x)\ge 0$ y $\sum_i w_i(x)=1$,
el voto ponderado de etiquetas 0/1 es un promedio convexo y queda en $[0,1]$. $k$-NN, el árbol y el bosque
cumplen esa condición; OLS y ridge suman 1 pero pueden tener pesos negativos.

### 0.3 ¿Por qué `casual` y `registered` no pueden ser variables?

In [4]:
print("¿casual + registered == bikers en todas las filas?",
      bool((bike.casual + bike.registered == bike.bikers).all()))

¿casual + registered == bikers en todas las filas? True


Porque son la respuesta partida en dos: `bikers = casual + registered` en el 100 % de las filas.
Usarlas sería *fuga de información* (target leakage): el modelo "predeciría" con la respuesta misma
y además, a la hora de usarlo (planear la reubicación de bicicletas por la mañana), esas cantidades
todavía no se conocen.

### 0.4 Desviación estándar antes de estandarizar
La guardamos en `desv_cruda`; la usamos en la Parte 4.

In [5]:
desv_cruda = Xb_raw_tr.std(ddof=0)
pd.DataFrame({"desv. estándar (entrenamiento, crudo)": desv_cruda,
              "media": Xb_raw_tr.mean()}).sort_values("desv. estándar (entrenamiento, crudo)", ascending=False)

,"desv. estándar (entrenamiento, crudo)",media
day,104.9296,184.6666
hr,6.8997,11.5759
mnth,3.4313,6.5805
weekday,2.0025,2.9935
weathersit,0.6549,1.4377
workingday,0.4649,0.6841
temp,0.1978,0.4894
hum,0.1968,0.6434
holiday,0.1668,0.0286
windspeed,0.1227,0.1904


---
## Parte 1: Los pesos de cinco modelos (California)

Si $\hat y(x)=\sum_i w_i(x) y_i$, en los puntos de entrenamiento $\hat{\mathbf y} = S\mathbf y$ con $S_{ij}=w_j(x_i)$.

### 1.1 OLS
$\hat\beta = (\tilde X^\top\tilde X)^{-1}\tilde X^\top \mathbf y$, entonces
$\hat y(x) = \tilde x^\top\hat\beta = \underbrace{\tilde x^\top (\tilde X^\top\tilde X)^{-1}\tilde X^\top}_{w(x)^\top}\mathbf y$.

**Suma de pesos.** $\sum_i w_i(x) = w(x)^\top\mathbf 1$. Como $\mathbf 1$ es la primera columna de $\tilde X$,
$\mathbf 1 = \tilde X e_1$ y
$$w(x)^\top \mathbf 1 = \tilde x^\top (\tilde X^\top\tilde X)^{-1}\tilde X^\top\tilde X e_1 = \tilde x^\top e_1 = 1 .$$
Equivalentemente, $H\mathbf 1 = \mathbf 1$: la matriz sombrero proyecta sobre el espacio columna de $\tilde X$,
y $\mathbf 1$ ya vive ahí, así que la proyección lo deja igual.

In [6]:
ols_c = LinearRegression().fit(Xc_tr, yc_tr)
W = P.pesos_ols(Xc_tr, Xc_te[:20])
print("max |W y - predict| =", np.abs(W @ yc_tr - ols_c.predict(Xc_te[:20])).max())
print("suma de pesos (min, max):", W.sum(1).min(), W.sum(1).max())

# H 1 = 1 sobre una submuestra (H completa de 16 512 x 16 512 no hace falta)
H_sub = P.pesos_ols(Xc_tr, Xc_tr[:500])
print("H[:500] @ 1 -> min, max:", H_sub.sum(1).min(), H_sub.sum(1).max())

max |W y - predict| = 4.04121180963557e-14
suma de pesos (min, max): 0.9999999999999997 1.0000000000000004
H[:500] @ 1 -> min, max: 0.9999999999999994 1.0000000000000007


### 1.2 Ridge
`Ridge` de scikit-learn no penaliza el intercepto. Resolver
$\min_{\beta_0,\beta}\ \|\mathbf y-\beta_0\mathbf 1-X\beta\|^2+\lambda\|\beta\|^2$:
la derivada respecto a $\beta_0$ da $\hat\beta_0=\bar y-\bar x^\top\hat\beta$, y sustituyendo queda un ridge
sobre datos centrados, $\hat\beta = (X_c^\top X_c+\lambda I)^{-1}X_c^\top\mathbf y$
(usamos $X_c^\top \mathbf 1 = 0$, de modo que $X_c^\top(\mathbf y - \bar y \mathbf 1) = X_c^\top \mathbf y$). Entonces
$$\hat y(x)=\bar y+(x-\bar x)^\top\hat\beta=\Big[\tfrac1n\mathbf 1^\top+(x-\bar x)^\top(X_c^\top X_c+\lambda I)^{-1}X_c^\top\Big]\mathbf y .$$
La suma es $1 + (x-\bar x)^\top(\cdots)X_c^\top\mathbf 1 = 1$ porque $X_c^\top\mathbf 1=0$.

In [7]:
LAM = 100
ridge_c = Ridge(alpha=LAM).fit(Xc_tr, yc_tr)
W = P.pesos_ridge(Xc_tr, Xc_te[:20], LAM)
print("ridge (intercepto libre): max |W y - predict| =", np.abs(W @ yc_tr - ridge_c.predict(Xc_te[:20])).max(),
      "| suma de pesos:", W.sum(1).min(), W.sum(1).max())

W_ing = P.pesos_ridge_ingenuo(Xc_tr, Xc_te[:20], LAM)
print("ridge 'obvio' (unos penalizados): max |W y - predict| =",
      np.abs(W_ing @ yc_tr - ridge_c.predict(Xc_te[:20])).max())
print("suma de pesos del ridge 'obvio':", W_ing.sum(1).min(), "a", W_ing.sum(1).max())
# reproduce exactamente un Ridge sin intercepto ajustado sobre [1, X]
r_unos = Ridge(alpha=LAM, fit_intercept=False).fit(P.con_unos(Xc_tr), yc_tr)
print("...pero sí reproduce Ridge(fit_intercept=False) sobre [1, X]:",
      np.abs(W_ing @ yc_tr - r_unos.predict(P.con_unos(Xc_te[:20]))).max())
print("intercepto: libre =", ridge_c.intercept_, "| penalizado =", r_unos.coef_[0], "| media de y =", yc_tr.mean())

ridge (intercepto libre): max |W y - predict| = 1.3322676295501878e-15 | suma de pesos: 0.9999999999999998 1.0
ridge 'obvio' (unos penalizados): max |W y - predict| = 0.012475914754048967
suma de pesos del ridge 'obvio': 0.9939802552371777 a 0.993980255237178
...pero sí reproduce Ridge(fit_intercept=False) sobre [1, X]: 2.220446049250313e-15
intercepto: libre = 2.072498958938959 | penalizado = 2.060023044184931 | media de y = 2.072498958938953


**¿Cuadra?** No. Al penalizar la columna de unos el intercepto se encoge hacia 0
($\hat\beta_0 = n\bar y/(n+\lambda)$ cuando las $X$ están centradas), así que las predicciones quedan
ligeramente sesgadas hacia 0. Los pesos ya no suman 1: con $X$ estandarizada ($\bar x=0$) la parte del
intercepto aporta $n/(n+\lambda)$ en vez de 1 (aquí $16512/16612\approx 0.99398$), y el resto suma
exactamente 0. Un voto ponderado que no suma 1 no es un promedio: si sumamos una constante $c$ a todas las
$y$, la predicción solo sube $c\cdot n/(n+\lambda)$. Por eso sklearn no penaliza el intercepto.

### 1.3 $k$-NN y árbol
* $k$-NN: $w_i(x) = \frac1k\,\mathbf 1\{x_i \in N_k(x)\}$, con $N_k(x)$ los $k$ vecinos más cercanos (`.kneighbors()`).
* Árbol: $w_i(x) = \dfrac{\mathbf 1\{x_i \text{ cae en la misma hoja que } x\}}{\#\{\text{filas de entrenamiento en esa hoja}\}}$ (`.apply()`).

Ambos son pesos no negativos que suman 1: promedios locales.

In [8]:
knn_c = KNeighborsRegressor(n_neighbors=30).fit(Xc_tr, yc_tr)
arbol_c = DecisionTreeRegressor(max_depth=6, random_state=RS).fit(Xc_tr, yc_tr)
for nombre, mod, Wf in [("k-NN", knn_c, P.pesos_knn(knn_c, Xc_te[:20], len(Xc_tr))),
                        ("árbol", arbol_c, P.pesos_arbol(arbol_c, Xc_tr, Xc_te[:20]))]:
    print(f"{nombre:6s} max |W y - predict| = {np.abs(Wf @ yc_tr - mod.predict(Xc_te[:20])).max():.2e}")

k-NN   max |W y - predict| = 8.88e-16
árbol  max |W y - predict| = 3.55e-15


### 1.4 *Random forest*
Con `bootstrap=False` cada árbol $t$ se entrena con **todas** las filas; su predicción es el promedio de su hoja,
$\hat y_t(x)=\sum_i \frac{\mathbf 1\{x_i\in L_t(x)\}}{|L_t(x)|}y_i$, y el bosque promedia los $T$ árboles:
$$\hat y(x)=\frac1T\sum_t\hat y_t(x)=\sum_i\underbrace{\frac1T\sum_{t}\frac{\mathbf 1\{x_i\in L_t(x)\}}{|L_t(x)|}}_{w_i(x)}y_i .$$

In [9]:
RF_KW = dict(n_estimators=200, min_samples_leaf=5, max_features=0.33, random_state=RS, n_jobs=-1)
rf_c = RandomForestRegressor(bootstrap=False, **RF_KW).fit(Xc_tr, yc_tr)
W = P.pesos_bosque(rf_c, Xc_tr, Xc_te[:20])
print("bootstrap=False: max |W y - predict| =", np.abs(W @ yc_tr - rf_c.predict(Xc_te[:20])).max())

rf_cb = RandomForestRegressor(bootstrap=True, **RF_KW).fit(Xc_tr, yc_tr)
W_b = P.pesos_bosque(rf_cb, Xc_tr, Xc_te[:20])
print("bootstrap=True, misma fórmula: max |W y - predict| =", np.abs(W_b @ yc_tr - rf_cb.predict(Xc_te[:20])).max())

C = P.conteos_bootstrap(rf_cb, len(Xc_tr))
W_bc = P.pesos_bosque(rf_cb, Xc_tr, Xc_te[:20], conteos=C)
print("bootstrap=True, contando cada fila tantas veces como entró a su árbol: max |W y - predict| =",
      np.abs(W_bc @ yc_tr - rf_cb.predict(Xc_te[:20])).max())
print("fracción media de filas fuera de la bolsa por árbol:", (C == 0).mean())

bootstrap=False: max |W y - predict| = 2.220446049250313e-15


bootstrap=True, misma fórmula: max |W y - predict| = 0.04396649376645989


bootstrap=True, contando cada fila tantas veces como entró a su árbol: max |W y - predict| = 2.6645352591003757e-15
fracción media de filas fuera de la bolsa por árbol: 0.36759447674418605


**¿Por qué ya no coincide?** Con `bootstrap=True` cada árbol se entrena con una muestra con reemplazo:
~36.8 % de las filas no entran (fuera de la bolsa) y otras entran 2, 3, … veces. El valor de cada hoja es el
promedio de **su muestra bootstrap**, no de todas las filas que caen en la hoja. Si reemplazamos el indicador por
el número de veces $c_{it}$ que la fila $i$ entró al árbol $t$,
$w_i(x)=\frac1T\sum_t \frac{c_{it}\,\mathbf 1\{x_i\in L_t(x)\}}{\sum_j c_{jt}\mathbf 1\{x_j\in L_t(x)\}}$,
el voto vuelve a reproducir `.predict()` exactamente (celda de arriba).

### 1.5 Verificación

In [10]:
modelos_c = {
    "OLS": (ols_c, lambda Xq: P.pesos_ols(Xc_tr, Xq)),
    "Ridge": (ridge_c, lambda Xq: P.pesos_ridge(Xc_tr, Xq, LAM)),
    "k-NN": (knn_c, lambda Xq: P.pesos_knn(knn_c, Xq, len(Xc_tr))),
    "Árbol": (arbol_c, lambda Xq: P.pesos_arbol(arbol_c, Xc_tr, Xq)),
    "Bosque": (rf_c, lambda Xq: P.pesos_bosque(rf_c, Xc_tr, Xq)),
}
for nombre, (mod, fpesos) in modelos_c.items():
    W = fpesos(Xc_te[:20])
    pred = mod.predict(Xc_te[:20])
    for q in range(20):
        P.verificar(W[q], yc_tr, pred[q])
    print(f"{nombre:7s}: 20/20 filas de prueba verificadas (|sum w y - pred| < 1e-8)")

FILA = 7
tabla = []
for nombre, (mod, fpesos) in modelos_c.items():
    w = fpesos(Xc_te[FILA:FILA + 1])[0]
    tabla.append({"modelo": nombre, "predicción": mod.predict(Xc_te[FILA:FILA + 1])[0], "sum w": w.sum(),
                  "pesos != 0": int(np.sum(np.abs(w) > 1e-15)), "pesos < 0": int(np.sum(w < -1e-15))})
print(f"Fila de prueba {FILA} (índice 0-based), y real = {yc_te[FILA]:.4f}")
pd.DataFrame(tabla).set_index("modelo")

OLS    : 20/20 filas de prueba verificadas (|sum w y - pred| < 1e-8)
Ridge  : 20/20 filas de prueba verificadas (|sum w y - pred| < 1e-8)
k-NN   : 20/20 filas de prueba verificadas (|sum w y - pred| < 1e-8)
Árbol  : 20/20 filas de prueba verificadas (|sum w y - pred| < 1e-8)


Bosque : 20/20 filas de prueba verificadas (|sum w y - pred| < 1e-8)


Fila de prueba 7 (índice 0-based), y real = 3.3690


,predicción,sum w,pesos != 0,pesos < 0
modelo,,,,
OLS,3.0566,1.0000,16512,4144
Ridge,3.0561,1.0000,16512,4130
k-NN,3.2082,1.0000,30,0
Árbol,3.6990,1.0000,328,0
Bosque,3.7180,1.0000,388,0


OLS y ridge usan **todas** las filas y casi la mitad con signo negativo; $k$-NN usa exactamente 30,
el árbol las de una sola hoja y el bosque la unión de las hojas de sus 200 árboles. Los cinco suman 1.

### 1.6 Proximidad

In [11]:
prox = P.proximidad(rf_c, Xc_tr, Xc_te[FILA:FILA + 1])[0]
w_rf = P.pesos_bosque(rf_c, Xc_tr, Xc_te[FILA:FILA + 1])[0]
tam_hojas = (rf_c.apply(Xc_tr) == rf_c.apply(Xc_te[FILA:FILA + 1])).sum(axis=0)
print("suma de la proximidad:", prox.sum())
print("tamaño medio de la hoja de x en los 200 árboles:", tam_hojas.mean())
print("filas con proximidad > 0:", (prox > 0).sum(), "| filas con w > 0:", (w_rf > 0).sum())
print("¿proximidad normalizada = w?  max |prox/sum(prox) - w| =", np.abs(prox / prox.sum() - w_rf).max())

suma de la proximidad: 6.495
tamaño medio de la hoja de x en los 200 árboles: 6.495
filas con proximidad > 0: 388 | filas con w > 0: 388
¿proximidad normalizada = w?  max |prox/sum(prox) - w| = 0.0025940971076652414


$\sum_i \text{prox}_i(x)=\frac1T\sum_t |L_t(x)|$: el **tamaño medio de la hoja** donde cae $x$, no 1.
No es un peso porque no está normalizada **dentro de cada árbol**: en la proximidad cada coincidencia vale 1,
así que una hoja grande "vota" más fuerte que una pequeña. Para ser el peso del bosque le falta dividir cada
coincidencia entre el tamaño de su hoja, $\frac{1}{|L_t(x)|}$, antes de promediar. Aun normalizándola al final
(dividir entre su suma) no coincide con $w$ porque las hojas tienen tamaños distintos en cada árbol.
